# PyHealth Trainer Tutorial — End-to-End Training with MIMIC-III

This notebook covers **`pyhealth.trainer`** — the training loop that ties datasets, models, and metrics together.

You will learn:
- How to load a public **synthetic MIMIC-III** dataset (no credentials required)
- How to apply a **mortality prediction task** to generate model-ready samples
- How to split, batch, and feed data to an **RNN model**
- How to use **`Trainer`** for training with validation, early stopping, and checkpointing
- How to **evaluate** a trained model on a held-out test set

> **Dataset Note:** The dataset used here is a fully synthetic MIMIC-III replica hosted by Google Cloud Storage. No PhysioNet account or data use agreement is needed.

---

In [ ]:
import torch
from pyhealth.datasets import MIMIC3Dataset, get_dataloader, split_by_patient
from pyhealth.models import RNN
from pyhealth.tasks import MortalityPredictionMIMIC3
from pyhealth.trainer import Trainer

---
## Step 1: Load the Synthetic MIMIC-III Dataset

`MIMIC3Dataset` loads structured EHR tables from a root directory (local path or URL). The Google Cloud Storage path below hosts a synthetic copy — the schema is identical to real MIMIC-III but no real patient data is present.

Default tables always loaded: `["patients", "admissions", "icustays"]`
Additional tables you can specify:
- `"diagnoses_icd"` — ICD-9 diagnosis codes per admission
- `"procedures_icd"` — ICD-9 procedure codes per admission
- `"prescriptions"` — Medication orders (NDC codes)
- `"labevents"` — Lab measurements
- `"noteevents"` — Clinical notes (discharge summaries, radiology reports, etc.)

In [ ]:
root = "https://storage.googleapis.com/pyhealth/Synthetic_MIMIC-III"

dataset = MIMIC3Dataset(
    root=root,
    dataset_name="mimic3",
    tables=[
        "diagnoses_icd",    # needed for mortality task (conditions)
        "procedures_icd",   # needed for mortality task (procedures)
        "prescriptions",    # needed for mortality task (drugs)
        "noteevents",       # clinical notes (not used by basic task, but loaded for reference)
    ],
)

print("Dataset loaded.")
print(f"  Number of patients: {len(dataset.patients)}")

---
## Step 2: Apply the Mortality Prediction Task

A **task** is a callable that transforms raw `Patient` objects into model-ready sample dicts. `MortalityPredictionMIMIC3` predicts whether a patient will die in the hospital during a subsequent admission.

**Task schema:**
```python
input_schema  = {"conditions": "sequence", "procedures": "sequence", "drugs": "sequence"}
output_schema = {"mortality": "binary"}
```

**Label definition:** `mortality = 1` if `hospital_expire_flag == 1` in the NEXT admission, else 0.

**Filtering:** Samples with no conditions, no procedures, OR no drugs are excluded.

`dataset.set_task(task)` iterates over all patients, calls the task on each, and returns a `SampleDataset` with fitted processors.

In [ ]:
task = MortalityPredictionMIMIC3()
samples = dataset.set_task(task)

print("Task applied.")
print(f"  Task name:     {samples.task_name}")
print(f"  Total samples: {len(samples)}")
print(f"  Input schema:  {samples.input_schema}")
print(f"  Output schema: {samples.output_schema}")

In [ ]:
# Inspect the fitted processors (tokenizers / label encoders)
print("Input processors:")
for key, proc in samples.input_processors.items():
    print(f"  {key}: {type(proc).__name__}, vocab size = {proc.size()}")

print("\nOutput processors:")
for key, proc in samples.output_processors.items():
    print(f"  {key}: {type(proc).__name__}, size = {proc.size()}")

---
## Step 3: Split the Dataset

`split_by_patient` partitions samples so that **no patient appears in more than one split** — this is the correct way to split clinical data. Splitting by sample (the naive approach) would allow data leakage: a model could see one visit from patient X in training and another visit from the same patient in test.

Returns three `SampleDataset` objects sharing the same fitted processors.

In [ ]:
train_ds, val_ds, test_ds = split_by_patient(
    dataset=samples,
    ratios=[0.7, 0.15, 0.15],
    seed=42,
)

print(f"Train:      {len(train_ds)} samples")
print(f"Validation: {len(val_ds)}  samples")
print(f"Test:       {len(test_ds)}  samples")

---
## Step 4: Create DataLoaders

`get_dataloader` wraps a `SampleDataset` in a `DataLoader` with PyHealth's custom `collate_fn_dict`, which handles variable-length sequences by padding and producing mask tensors.

In [ ]:
train_loader = get_dataloader(train_ds, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val_ds,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test_ds,  batch_size=32, shuffle=False)

print(f"Train batches: {len(train_loader)}")
print(f"Val   batches: {len(val_loader)}")
print(f"Test  batches: {len(test_loader)}")

# Inspect a batch
batch = next(iter(train_loader))
print("\nBatch keys:", list(batch.keys()))

---
## Step 5: Build the Model

The `RNN` model takes the `SampleDataset` as its first argument. It reads:
- `dataset.input_schema` to know which features exist and their types
- `dataset.input_processors` to know vocabulary sizes (for embedding tables)
- `dataset.output_schema` to determine the number of output classes and the loss function

You never need to specify vocabulary sizes or output sizes manually.

In [ ]:
model = RNN(
    dataset=samples,       # pass the full SampleDataset (with fitted processors)
    embedding_dim=64,
    hidden_dim=64,
    rnn_type="GRU",
    num_layers=1,
    dropout=0.3,
)

print(model)
print()
print(f"Feature keys:   {model.feature_keys}")
print(f"Label key:      {model.label_key}")
print(f"Mode:           {model.mode}")
print(f"Output size:    {model.get_output_size()}")
print(f"Loss function:  {model.get_loss_function()}")

---
## Step 6: Setup the Trainer

```python
Trainer(
    model,
    checkpoint_path = None,         # load from checkpoint if provided
    metrics         = None,         # default metrics per mode (e.g., pr_auc, roc_auc, f1)
    device          = None,         # auto-detects CUDA; falls back to CPU
    enable_logging  = True,         # writes log.txt to output_path/exp_name/
    output_path     = "./output",   # directory for logs and checkpoints
    exp_name        = None,         # defaults to current datetime string
)
```

`Trainer` automatically moves the model to the detected device.

In [ ]:
trainer = Trainer(
    model=model,
    output_path="./output",
    exp_name="mortality_rnn_demo",
    enable_logging=True,
)

print(f"Training on device: {trainer.device}")
print(f"Logs written to:    {trainer.exp_path}")

---
## Step 7: Train the Model

```python
trainer.train(
    train_dataloader,
    val_dataloader     = None,
    test_dataloader    = None,
    epochs             = 5,
    optimizer_class    = torch.optim.Adam,
    optimizer_params   = {"lr": 1e-3},
    weight_decay       = 0.0,
    max_grad_norm      = None,        # gradient clipping (e.g., 1.0)
    monitor            = None,        # metric name to track for best model
    monitor_criterion  = "max",       # "max" or "min"
    load_best_model_at_last = True,   # restore best weights after training
    patience           = None,        # early stopping patience in epochs
)
```

**Monitoring:** When `monitor` is set, Trainer saves the checkpoint with the best value of that metric on the validation set, and optionally restores it at the end of training.

**Early stopping:** Set `patience=N` to stop training if the monitored metric does not improve for N consecutive evaluation epochs.

In [ ]:
trainer.train(
    train_dataloader=train_loader,
    val_dataloader=val_loader,
    test_dataloader=test_loader,
    epochs=5,
    optimizer_class=torch.optim.Adam,
    optimizer_params={"lr": 1e-3},
    weight_decay=1e-4,
    monitor="roc_auc",          # track ROC-AUC on validation set
    monitor_criterion="max",    # higher ROC-AUC = better
    load_best_model_at_last=True,
    patience=3,                 # stop if no improvement for 3 epochs
)

---
## Step 8: Evaluate on the Test Set

`trainer.evaluate(dataloader)` runs inference on the given dataloader and returns a dict of metric scores using the model's default metric function for its mode.

For binary classification the defaults are: `pr_auc`, `roc_auc`, `f1`.

In [ ]:
test_metrics = trainer.evaluate(test_loader)

print("Test set results:")
for metric, value in test_metrics.items():
    print(f"  {metric:15s}: {value:.4f}")

---
## Step 9: Checkpoint Saving and Loading

PyHealth's Trainer saves checkpoints as standard PyTorch `.pt` files containing the `model.state_dict()`.

In [ ]:
import os

# Save manually
ckpt_path = "./output/mortality_rnn_demo/manual_checkpoint.pt"
os.makedirs(os.path.dirname(ckpt_path), exist_ok=True)
trainer.save_ckpt(ckpt_path)
print(f"Checkpoint saved to: {ckpt_path}")

In [ ]:
# Load a checkpoint into an existing Trainer / model
# This is useful for resuming training or loading a pretrained model for inference
new_model = RNN(dataset=samples, embedding_dim=64, hidden_dim=64)

new_trainer = Trainer(
    model=new_model,
    checkpoint_path=ckpt_path,   # loads weights immediately on init
    enable_logging=False,
)

# Verify the loaded model produces the same test metrics
loaded_metrics = new_trainer.evaluate(test_loader)
print("Metrics from loaded checkpoint:")
for metric, value in loaded_metrics.items():
    print(f"  {metric:15s}: {value:.4f}")

---
## Step 10: Raw Inference with `trainer.inference()`

If you need the raw predictions (not just aggregated metrics), use `trainer.inference()`. This is useful for computing custom metrics, error analysis, or feeding into a calibration step.

In [ ]:
y_true_all, y_prob_all, loss_mean = trainer.inference(test_loader)

print(f"y_true shape: {y_true_all.shape}")
print(f"y_prob shape: {y_prob_all.shape}")
print(f"Mean test loss: {loss_mean:.4f}")
print()
print("First 10 predictions:")
for i in range(min(10, len(y_true_all))):
    print(f"  true={int(y_true_all[i])}  pred_prob={y_prob_all[i].item():.3f}")

---
## Summary

### Full pipeline at a glance

```python
# 1. Load EHR data
dataset = MIMIC3Dataset(root=root, tables=["diagnoses_icd", "procedures_icd", "prescriptions"])

# 2. Define prediction task
samples = dataset.set_task(MortalityPredictionMIMIC3())

# 3. Split by patient (prevents leakage)
train_ds, val_ds, test_ds = split_by_patient(samples, ratios=[0.7, 0.15, 0.15])

# 4. Create DataLoaders
train_loader = get_dataloader(train_ds, batch_size=32, shuffle=True)

# 5. Build model (reads schema and processors from dataset)
model = RNN(dataset=samples, embedding_dim=64, hidden_dim=64)

# 6. Train
trainer = Trainer(model, output_path="./output", exp_name="my_run")
trainer.train(train_loader, val_loader, monitor="roc_auc", epochs=20, patience=5)

# 7. Evaluate
trainer.evaluate(test_loader)
```

### Key Trainer parameters

| Parameter | Effect |
|-----------|--------|
| `monitor` | Metric name to track for best model checkpoint |
| `monitor_criterion` | `"max"` for metrics like AUC/F1; `"min"` for loss |
| `patience` | Early stopping: stop after N epochs without improvement |
| `load_best_model_at_last` | Restore best checkpoint weights after training ends |
| `weight_decay` | L2 regularization (passed to optimizer) |
| `max_grad_norm` | Gradient clipping (useful for LSTM stability) |

---
## API Reference: Available Metric Strings

The `metrics` argument to `Trainer.__init__` and the `monitor` argument to `trainer.train()` are plain strings drawn from a fixed list. The exact list depends on **`model.mode`**, which is set automatically from the task's output schema:

```python
print(model.mode)   # → "binary" | "multiclass" | "multilabel" | "regression"
```

`Trainer` uses `model.mode` to select the right metrics function, then passes your `metrics` list to it. Any string you pass to `monitor` must appear in that same list — otherwise evaluation will raise a `KeyError`.

To compute a non-default set of metrics and track a specific one:
```python
trainer = Trainer(
    model=model,
    metrics=["roc_auc", "pr_auc", "balanced_accuracy", "ECE"],  # computed every eval epoch
)
trainer.train(..., monitor="pr_auc", monitor_criterion="max")
```

---

### Binary classification — `mode = "binary"`
**Source:** `pyhealth.metrics.binary_metrics_fn`  
**Defaults when `metrics=None`:** `["pr_auc", "roc_auc", "f1"]`

| Metric string | Description | `monitor_criterion` |
|---|---|---|
| `"pr_auc"` | Area under the Precision-Recall curve | `"max"` |
| `"roc_auc"` | Area under the ROC curve | `"max"` |
| `"f1"` | F1 score at `threshold` (default 0.5) | `"max"` |
| `"accuracy"` | Fraction of correct predictions | `"max"` |
| `"balanced_accuracy"` | Accuracy adjusted for class imbalance | `"max"` |
| `"precision"` | Precision at `threshold` | `"max"` |
| `"recall"` | Recall at `threshold` | `"max"` |
| `"cohen_kappa"` | Cohen's kappa (agreement beyond chance) | `"max"` |
| `"jaccard"` | Jaccard similarity coefficient | `"max"` |
| `"ECE"` | Expected Calibration Error (20 equal-width bins) | `"min"` |
| `"ECE_adapt"` | Adaptive ECE (20 equal-size bins) | `"min"` |

---

### Multiclass classification — `mode = "multiclass"`
**Source:** `pyhealth.metrics.multiclass_metrics_fn`  
**Defaults when `metrics=None`:** `["accuracy", "f1_macro", "f1_micro"]`

| Metric string | Description | `monitor_criterion` |
|---|---|---|
| `"accuracy"` | Overall accuracy | `"max"` |
| `"balanced_accuracy"` | Accuracy adjusted for class imbalance | `"max"` |
| `"f1_macro"` | F1, macro-averaged across classes | `"max"` |
| `"f1_micro"` | F1, micro-averaged across classes | `"max"` |
| `"f1_weighted"` | F1, weighted by class support | `"max"` |
| `"roc_auc_macro_ovo"` | ROC-AUC, macro, one-vs-one | `"max"` |
| `"roc_auc_macro_ovr"` | ROC-AUC, macro, one-vs-rest | `"max"` |
| `"roc_auc_weighted_ovo"` | ROC-AUC, weighted, one-vs-one | `"max"` |
| `"roc_auc_weighted_ovr"` | ROC-AUC, weighted, one-vs-rest | `"max"` |
| `"jaccard_micro"` | Jaccard, micro-averaged | `"max"` |
| `"jaccard_macro"` | Jaccard, macro-averaged | `"max"` |
| `"jaccard_weighted"` | Jaccard, weighted | `"max"` |
| `"cohen_kappa"` | Cohen's kappa | `"max"` |
| `"brier_top1"` | Brier score for the top predicted class | `"min"` |
| `"ECE"` | Expected Calibration Error (20 equal-width bins) | `"min"` |
| `"ECE_adapt"` | Adaptive ECE (20 equal-size bins) | `"min"` |
| `"cwECEt"` | Classwise ECE with threshold = min(0.01, 1/K) | `"min"` |
| `"cwECEt_adapt"` | Classwise adaptive ECE | `"min"` |
| `"hits@n"` | HITS@1 / HITS@5 / HITS@10 (produces 3 dict keys) | `"max"` |
| `"mean_rank"` | Mean rank + mean reciprocal rank | `"min"` |

---

### Multilabel classification — `mode = "multilabel"`
**Source:** `pyhealth.metrics.multilabel_metrics_fn`  
**Defaults when `metrics=None`:** `["pr_auc_samples"]`  
**Note:** threshold defaults to `0.3` (not `0.5`) — lower thresholds are common in drug recommendation tasks.

| Metric string | Description | `monitor_criterion` |
|---|---|---|
| `"pr_auc_samples"` | PR-AUC, averaged across samples | `"max"` |
| `"pr_auc_micro"` | PR-AUC, micro-averaged | `"max"` |
| `"pr_auc_macro"` | PR-AUC, macro-averaged | `"max"` |
| `"pr_auc_weighted"` | PR-AUC, weighted | `"max"` |
| `"roc_auc_samples"` | ROC-AUC, samples-averaged | `"max"` |
| `"roc_auc_micro"` | ROC-AUC, micro-averaged | `"max"` |
| `"roc_auc_macro"` | ROC-AUC, macro-averaged | `"max"` |
| `"roc_auc_weighted"` | ROC-AUC, weighted | `"max"` |
| `"f1_samples"` | F1, samples-averaged | `"max"` |
| `"f1_micro"` | F1, micro-averaged | `"max"` |
| `"f1_macro"` | F1, macro-averaged | `"max"` |
| `"f1_weighted"` | F1, weighted | `"max"` |
| `"precision_micro"` / `"_macro"` / `"_weighted"` / `"_samples"` | Precision variants | `"max"` |
| `"recall_micro"` / `"_macro"` / `"_weighted"` / `"_samples"` | Recall variants | `"max"` |
| `"jaccard_micro"` / `"_macro"` / `"_weighted"` / `"_samples"` | Jaccard variants | `"max"` |
| `"accuracy"` | Element-wise accuracy | `"max"` |
| `"hamming_loss"` | Hamming loss | `"min"` |
| `"ddi"` | Drug-drug interaction rate (drug recommendation only) | `"min"` |
| `"cwECE"` | Classwise ECE (20 equal-width bins) | `"min"` |
| `"cwECE_adapt"` | Classwise adaptive ECE | `"min"` |

---

### Regression — `mode = "regression"`
**Source:** `pyhealth.metrics.regression_metrics_fn`  
**Defaults when `metrics=None`:** `["kl_divergence", "mse", "mae"]`

| Metric string | Description | `monitor_criterion` |
|---|---|---|
| `"mae"` | Mean Absolute Error | `"min"` |
| `"mse"` | Mean Squared Error | `"min"` |
| `"kl_divergence"` | KL divergence between true and reconstructed distributions | `"min"` |